1. Preprocessing

1.1 Image Resize (512x512)

In [5]:
import os
import tensorflow as tf
import glob

# Define the file path for images (add a wildcard for all image files)
image_filename = "E:/Sem 5/mini project/water_body/Water Bodies Dataset/post images mask/*"

# Create a list of image files
img_filename_list = sorted(glob.glob(image_filename))

# Check the count
if not img_filename_list:
    print(f"No images found in the directory: {image_filename}")
else:
    print(f"Found {len(img_filename_list)} images.")

# Create output directory for resized images
resized_image_dir = "E:/Sem 5/mini project/water_body/Resized Dataset/post images masks"
os.makedirs(resized_image_dir, exist_ok=True)

# Function to decode and save resized images
def decode_and_save_resized_images(img_filename, output_dir):
    try:
        # Read the image file
        image_string = tf.io.read_file(img_filename)

        # Decode the image
        image = tf.image.decode_jpeg(image_string, channels=3)

        # Resize the image
        image = tf.image.resize(image, (512, 512))

        # Extract the original image name
        img_name = os.path.basename(img_filename)

        # Save the resized image with the original name
        img_save_path = os.path.join(output_dir, img_name)
        tf.keras.preprocessing.image.save_img(img_save_path, image.numpy())
        print(f"Resized and saved image: {img_save_path}")
    except Exception as e:
        print(f"Error processing file {img_filename}: {e}")

# Process and save all resized images
for img_file in img_filename_list:
    decode_and_save_resized_images(img_file, resized_image_dir)

print(f"Resized images saved in: {resized_image_dir}")


Found 2841 images.
Resized and saved image: E:/Sem 5/mini project/water_body/Resized Dataset/post images masks\water_body_1.jpg
Resized and saved image: E:/Sem 5/mini project/water_body/Resized Dataset/post images masks\water_body_10.jpg
Resized and saved image: E:/Sem 5/mini project/water_body/Resized Dataset/post images masks\water_body_100.jpg
Resized and saved image: E:/Sem 5/mini project/water_body/Resized Dataset/post images masks\water_body_1000.jpg
Resized and saved image: E:/Sem 5/mini project/water_body/Resized Dataset/post images masks\water_body_1002.jpg
Resized and saved image: E:/Sem 5/mini project/water_body/Resized Dataset/post images masks\water_body_1003.jpg
Resized and saved image: E:/Sem 5/mini project/water_body/Resized Dataset/post images masks\water_body_1004.jpg
Resized and saved image: E:/Sem 5/mini project/water_body/Resized Dataset/post images masks\water_body_1006.jpg
Resized and saved image: E:/Sem 5/mini project/water_body/Resized Dataset/post images masks

1.2 Normalization

In [ ]:
import os
import tensorflow as tf
import glob
import numpy as np

# Define the file path for resized images
resized_image_filename = "D:/mini project/resized_images2/train/*.jpg"

# Create a list for resized images
resized_img_filename_list = sorted(glob.glob(resized_image_filename))

# Check the count
print(len(resized_img_filename_list))

# Create output directories
normalized_image_dir = "D:/mini project/normalized_images5/images/train"
pixel_values_dir_nor = "D:/mini project/normalized_images5/pixel_values_txt_nor/train"
pixel_values_dir_org = "D:/mini project/normalized_images5/pixel_values_txt_org/train"
os.makedirs(normalized_image_dir, exist_ok=True)
os.makedirs(pixel_values_dir_nor, exist_ok=True)
os.makedirs(pixel_values_dir_org, exist_ok=True)

# Function to normalize, save images, and save pixel values
def normalize_and_save_images(img_filename, normalized_dir, pixel_values_dir_nor,pixel_values_dir_org):
    image_string = tf.io.read_file(img_filename)

    # Decode the image
    image = tf.image.decode_jpeg(image_string, channels=3)

    # Save pixel values before normalization
    original_pixels = image.numpy()

    # Normalize the image (convert to float and scale to [0, 1])
    normalized_image = tf.image.convert_image_dtype(image, tf.float32)

    # Save pixel values after normalization
    normalized_pixels = normalized_image.numpy()

    # Extract the original image name
    img_name = os.path.basename(img_filename)
    img_base_name = os.path.splitext(img_name)[0]

    # Save the normalized image
    normalized_save_path = os.path.join(normalized_dir, img_name)
    tf.keras.preprocessing.image.save_img(normalized_save_path, normalized_image.numpy())

    # Save pixel values to .txt files
    pixel_values_original_path = os.path.join(pixel_values_dir_org, f"{img_base_name}_original.txt")
    pixel_values_normalized_path = os.path.join(pixel_values_dir_nor, f"{img_base_name}_normalized.txt")
    np.savetxt(pixel_values_original_path, original_pixels.reshape(-1, original_pixels.shape[-1]), fmt='%d', header="Original Pixel Values")
    np.savetxt(pixel_values_normalized_path, normalized_pixels.reshape(-1, normalized_pixels.shape[-1]), fmt='%.6f', header="Normalized Pixel Values")

    # Print pixel values (for demonstration)
    print(f"Image: {img_name}")
    print(f"Original Pixels (sample): {original_pixels[0, :5]}")  # First row, first 5 pixels
    print(f"Normalized Pixels (sample): {normalized_pixels[0, :5]}")  # First row, first 5 pixels

# Process and save all normalized images and pixel values
for img_file in resized_img_filename_list:
    normalize_and_save_images(img_file, normalized_image_dir, pixel_values_dir_nor,pixel_values_dir_org)

print(f"Normalized images saved in: {normalized_image_dir}")
print(f"Pixel values saved in: {pixel_values_dir}")

2. Post Flood Map Generation

In [ ]:
import cv2
import numpy as np
import os

def detect_water_body(image):
    """
    Detect water bodies in an image based on color segmentation.

    Args:
        image (numpy.ndarray): Input image.

    Returns:
        water_mask (numpy.ndarray): Binary mask of water bodies.
    """
    # Convert the image to HSV for better color segmentation
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Define the range for detecting water (blue color)
    lower_blue = np.array([100, 150, 50])
    upper_blue = np.array([140, 255, 255])

    # Create a mask for water regions
    water_mask = cv2.inRange(hsv_image, lower_blue, upper_blue)
    return water_mask

def extend_water_body(water_mask, extent=10):
    """
    Extend detected water bodies using morphological operations.

    Args:
        water_mask (numpy.ndarray): Binary mask of water bodies.
        extent (int): Radius of extension in pixels.

    Returns:
        extended_mask (numpy.ndarray): Extended water mask.
    """
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (extent, extent))
    extended_mask = cv2.dilate(water_mask, kernel, iterations=1)
    return extended_mask

def process_image(image_path, output_path, flood_extent=10):
    """
    Process an image to detect and extend water bodies.

    Args:
        image_path (str): Path to the input image.
        output_path (str): Path to save the output image.
        flood_extent (int): Radius of water body extension.
    """
    # Load the input image
    image = cv2.imread(image_path)

    if image is None:
        raise FileNotFoundError(f"Image at path {image_path} could not be loaded.")

    # Detect water bodies
    water_mask = detect_water_body(image)

    # Extend the water bodies
    extended_mask = extend_water_body(water_mask, extent=flood_extent)

    # Create output image
    result = image.copy()
    result[extended_mask > 0] = [255, 0, 0]  # Highlight water regions in blue

    # Save the result
    cv2.imwrite(output_path, result)

# Example usage
if __name__ == "__main__":
    input_folder_path = "D:/mini project/Water Bodies Dataset/Images"
    output_folder_path = "D:/mini project/Water Bodies Dataset/post Images"

    # Ensure the output folder exists
    os.makedirs(output_folder_path, exist_ok=True)

    for filename in os.listdir(input_folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            input_image_path = os.path.join(input_folder_path, filename)
            output_image_path = os.path.join(output_folder_path, filename)

            try:
                process_image(input_image_path, output_image_path, flood_extent=10)
                print(f"Processed and saved: {output_image_path}")
            except FileNotFoundError as e:
                print(e)


2.1 Post Flood Mask Generation 

In [ ]:
import cv2
import numpy as np
import os

# Directory containing the images
input_dir = 'D:/mini project/Water Bodies Dataset/post Images/'
output_dir = 'D:/mini project/Water Bodies Dataset/post Images mask/'

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Loop through all files in the input directory
for filename in os.listdir(input_dir):
    # Get the full path to the image
    image_path = os.path.join(input_dir, filename)

    # Check if the file is an image (you can add more extensions as needed)
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        # Load the image
        image = cv2.imread(image_path)

        # Convert the image to the HSV color space
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        # Define the blue color range in HSV
        lower_blue = np.array([100, 150, 0])
        upper_blue = np.array([140, 255, 255])

        # Create a mask for detecting blue regions
        mask = cv2.inRange(hsv, lower_blue, upper_blue)

        # Create a black image of the same size as the input image
        output = np.zeros_like(image)

        # Set the blue regions to white in the output image
        output[mask > 0] = [255, 255, 255]

        # Save the output image with the same name
        output_path = os.path.join(output_dir, filename)
        cv2.imwrite(output_path, output)

        # Optionally display the image
        # cv2.imshow(f'Result: {filename}', output)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()

print("Processing complete.")


3. Water Body Segmentation using UNET Attention Model

In [ ]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
import glob

In [ ]:
images = sorted(glob.glob('/content/dataset all/water_bodies/*.jpg'))
masks = sorted(glob.glob('/content/dataset all/mask/*.jpg'))
len(images), len(masks)

In [ ]:
import numpy as np
import cv2
import os
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

class CustomDataGenerator(keras.utils.Sequence):
    def __init__(self, images, masks, batch_size=8, img_size=(512, 512), shuffle=True):
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.images = images #os.listdir(image_folder)
        self.masks = masks #os.listdir(mask_folder)

        # on each epoch end, shuffle the dataset
        self.on_epoch_end()

        # datagen function to augment the input image and mask pair
        self.datagen = ImageDataGenerator(
            rotation_range=5,
            width_shift_range=0.1,
            height_shift_range=0.1,
            zoom_range=0.05,
            horizontal_flip=True,
            vertical_flip=True,
            fill_mode = 'constant',
            cval=0.0,
        )

    # randomly crop the images to 512x512 size
    def random_crop(self, image, mask, crop_size=512):

        # image width and height calculation
        img_height, img_width = image.shape[0], image.shape[1]
        mask_height, mask_width = mask.shape[0], mask.shape[1]

        # random x and y coordinate for cropping the image
        x = np.random.randint(0, img_width - crop_size)
        y = np.random.randint(0, img_height - crop_size)

        # random crop
        image_crop = image[y:y + crop_size, x:x + crop_size, :]
        mask_crop = mask[y:y + crop_size, x:x + crop_size]

        return image_crop, mask_crop

    # data augmentation using keras ImageDataGenerator function
    def data_augmentation(self, image, mask):
        trans_param = self.datagen.get_random_transform(image.shape)
        image = self.datagen.apply_transform(image, trans_param)
        mask = self.datagen.apply_transform(mask, trans_param)
        return image, mask

    # length of the processing batch
    def __len__(self):
        return int(np.ceil(len(self.images) / self.batch_size))

    # data normalization
    def data_normalization(self, image, mask):

        # reshape mask from 512x512 to 512x512x1
        mask = mask.reshape((*self.img_size, 1))

        # Binary mask
        mask = np.where(mask<127, 0, 1)

        # data normalization (If you want to normalize another way, change the below line)
        image = image / 255.0

        # return image and mask
        return image, mask

    # data preprocessing, resize, crop image etc
    def data_preprocessing(self, image, mask):
        image, mask = cv2.resize(image, (576, 576)), cv2.resize(mask, (576, 576))
        image, mask = self.random_crop(image, mask)
        return image, mask

    # on each epoch, shuffle the dataset (image and mask index)
    def on_epoch_end(self):
        self.indexes = np.arange(len(self.images))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    # get item is the core function
    # this function will run in each batch/epoch to load the dataset into RAM and pass to DL model
    def __getitem__(self, index):

        # start and end index
        # the last index can be shorter than the number of batches
        start_idx = index * self.batch_size
        end_idx = min((index + 1) * self.batch_size, len(self.images))
        indexes = self.indexes[start_idx:end_idx]

        # initialize the images and mask batches
        batch_images = []
        batch_masks = []

        # iterate over each indexes in batch
        for i in indexes:
            img_path = self.images[i]
            mask_path = self.masks[i]

            # read image using open cv
            img = cv2.imread(img_path)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            # Skip if image or mask is not loaded properly
            if img is None or mask is None:
                continue

            # image preprocessing; resize, random crop
            img, mask = self.data_preprocessing(img, mask)

            # data normalization
            img, mask = self.data_normalization(img, mask)

            # data augmentation
            img, mask = self.data_augmentation(img, mask)

            # to fix the issue during training process
            mask = mask.astype(np.float32)

            # append each image, mask pair to the batches
            batch_images.append(img)
            batch_masks.append(mask)

        # return batch image and batch mamks as a numpy array (n, tile_x, tile_y, channels)
        return np.array(batch_images), np.array(batch_masks)

In [ ]:
import matplotlib.pyplot as plt

data = CustomDataGenerator(images, masks)
batch_images, batch_masks = data.__getitem__(0)

img = np.random.randint(0,8)
# Visualize the first image and its mask from the batch
image = batch_images[img]
mask = batch_masks[img]

# Plotting the image and its mask
plt.figure(figsize=(10, 5))

# Display Image
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title('Image')
plt.axis('off')

# Display Mask
plt.subplot(1, 2, 2)
plt.imshow(mask)
plt.title('Mask')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
images = sorted(glob.glob('/content/dataset all/water_bodies/*.jpg'))
masks = sorted(glob.glob('/content/dataset all/mask/*.jpg'))
print(f"Number of images found: {len(images)}")
print(f"Number of masks found: {len(masks)}")
print(f"Image paths (first 5): {images[:5]}")
print(f"Mask paths (first 5): {masks[:5]}") # Likely to be empty, causing the error.

In [ ]:
from sklearn.model_selection import train_test_split
train_img, test_img, train_mask, test_mask = train_test_split(images, masks, test_size=0.2, random_state=42)

In [ ]:
len(train_img), len(test_img)

In [ ]:
train_dataset = CustomDataGenerator(train_img, train_mask)
test_dataset = CustomDataGenerator(test_img, test_mask)

In [ ]:
len(train_dataset), len(test_dataset)

In [ ]:
import tensorflow as tf

# recall
def recall_m(y_true, y_pred):
    true_positives = tf.keras.backend.sum(tf.keras.backend.round(tf.keras.backend.clip(y_true * y_pred, 0, 1)))
    possible_positives = tf.keras.backend.sum(tf.keras.backend.round(tf.keras.backend.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + tf.keras.backend.epsilon())
    return recall

# precision
def precision_m(y_true, y_pred):
    true_positives = tf.keras.backend.sum(tf.keras.backend.round(tf.keras.backend.clip(y_true * y_pred, 0, 1)))
    predicted_positives = tf.keras.backend.sum(tf.keras.backend.round(tf.keras.backend.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + tf.keras.backend.epsilon())
    return precision

#f1 score
def f1_m(y_true, y_pred):
    precision = precision_m(y_true, y_pred)
    recall = recall_m(y_true, y_pred)
    return 2*((precision*recall)/(precision+recall+tf.keras.backend.epsilon()))

def dsc(y_true, y_pred):
    smooth = 1.
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    score = (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)
    return score

def dice_loss(y_true, y_pred):
    loss = 1 - dsc(y_true, y_pred)
    return loss

In [ ]:
import tensorflow as tf
# from keras import backend as K # Remove this line
#from keras.models import * # Change to tf.keras
from tensorflow.keras.models import * # Use tensorflow.keras consistently
from tensorflow.keras.layers import * # Use tensorflow.keras consistently
from tensorflow.keras.optimizers import * # Use tensorflow.keras consistently
from tensorflow.keras.losses import * # Use tensorflow.keras consistently
# from utils.utils import f1_m, precision_m, recall_m, dsc

from sklearn.metrics import *
# ... (rest of your model definition code) ...

def expend_as(tensor, rep,name):
	my_repeat = Lambda(lambda x, repnum: K.repeat_elements(x, repnum, axis=3), arguments={'repnum': rep},  name='psi_up'+name)(tensor)
	return my_repeat


def AttnGatingBlock(x, g, inter_shape, name):
    ''' take g which is the spatially smaller signal, do a conv to get the same
    number of feature channels as x (bigger spatially)
    do a conv on x to also get same geature channels (theta_x)
    then, upsample g to be same size as x
    add x and g (concat_xg)
    relu, 1x1 conv, then sigmoid then upsample the final - this gives us attn coefficients'''

    # Use x.shape instead of K.int_shape(x)
    shape_x = x.shape  # 32
    shape_g = g.shape  # 16

    theta_x = Conv2D(inter_shape, (2, 2), strides=(2, 2), padding='same', name='xl'+name)(x)  # 16
    # Use theta_x.shape instead of K.int_shape(theta_x)
    shape_theta_x = theta_x.shape

    phi_g = Conv2D(inter_shape, (1, 1), padding='same')(g)
    upsample_g = Conv2DTranspose(inter_shape, (3, 3),strides=(shape_theta_x[1] // shape_g[1], shape_theta_x[2] // shape_g[2]),padding='same', name='g_up'+name)(phi_g)  # 16

    concat_xg = add([upsample_g, theta_x])
    act_xg = Activation('relu')(concat_xg)
    psi = Conv2D(1, (1, 1), padding='same', name='psi'+name)(act_xg)
    sigmoid_xg = Activation('sigmoid')(psi)
    # Use sigmoid_xg.shape instead of K.int_shape(sigmoid_xg)
    shape_sigmoid = sigmoid_xg.shape
    upsample_psi = UpSampling2D(size=(shape_x[1] // shape_sigmoid[1], shape_x[2] // shape_sigmoid[2]))(sigmoid_xg)  # 32

    # upsample_psi = expend_as(upsample_psi, shape_x[3],  name)
    y = multiply([upsample_psi, x], name='q_attn'+name)

    result = Conv2D(shape_x[3], (1, 1), padding='same',name='q_attn_conv'+name)(y)
    result_bn = BatchNormalization(name='q_attn_bn'+name)(result)
    return result_bn

def UnetConv2D(input, outdim, is_batchnorm, name):
	x = Conv2D(outdim, (3, 3), strides=(1, 1), kernel_initializer=kinit, padding="same", name=name+'_1')(input)
	if is_batchnorm:
		x =BatchNormalization(name=name + '_1_bn')(x)
	x = Activation('relu',name=name + '_1_act')(x)

	x = Conv2D(outdim, (3, 3), strides=(1, 1), kernel_initializer=kinit, padding="same", name=name+'_2')(x)
	if is_batchnorm:
		x = BatchNormalization(name=name + '_2_bn')(x)
	x = Activation('relu', name=name + '_2_act')(x)
	return x


def UnetGatingSignal(input, is_batchnorm, name):
    ''' this is simply 1x1 convolution, bn, activation '''
    # shape = K.int_shape(input) # Replace this line
    shape = input.shape # With this line to use the static shape of the tensor
    x = Conv2D(shape[3] * 1, (1, 1), strides=(1, 1), padding="same", kernel_initializer=kinit, name=name + '_conv')(input)
    if is_batchnorm:
        x = BatchNormalization(name=name + '_bn')(x)
    x = Activation('relu', name=name + '_act')(x)
    return x

K.set_image_data_format('channels_last')  # TF dimension ordering in this code
kinit = 'glorot_normal'

def attn_unet(lr, loss_func=None, pretrained_weights = None, input_size = (256, 256, 6)):
        inputs = Input(shape=input_size)

        #Contraction path
        conv1 = UnetConv2D(inputs, 64, is_batchnorm=True, name='conv1')
        pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

        conv2 = UnetConv2D(pool1, 64, is_batchnorm=True, name='conv2')
        pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

        conv3 = UnetConv2D(pool2, 128, is_batchnorm=True, name='conv3')
        conv3 = Dropout(0.1,name='drop_conv3')(conv3)
        pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

        conv4 = UnetConv2D(pool3, 128, is_batchnorm=True, name='conv4')
        # conv4 = Dropout(0.2, name='drop_conv4')(conv4)
        pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)

        conv5 = UnetConv2D(pool4, 256, is_batchnorm=True, name='conv5')
        # conv5 = Dropout(0.2,name='drop_conv5')(conv5)
        pool5 = MaxPooling2D(pool_size=(2, 2))(conv5)

        conv6 = UnetConv2D(pool5, 256, is_batchnorm=True, name='conv6')
        # conv6 = Dropout(0.2, name='drop_conv6')(conv6)
        pool6 = MaxPooling2D(pool_size=(2, 2))(conv6)

        #center layer
        center = UnetConv2D(pool6, 512, is_batchnorm=True, name='center')

        # Expansion path
        g1 = UnetGatingSignal(center, is_batchnorm=True, name='g1')
        attn1 = AttnGatingBlock(conv6, g1, 512, '_1')
        # attn1 = Dropout(0.2, name='drop_attn1')(attn1)
        convt1 = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', activation='relu', kernel_initializer=kinit, name='convt1')(center)
        up1 = concatenate([convt1, attn1], name='up1')

        g2 = UnetGatingSignal(up1, is_batchnorm=True, name='g2')
        attn2 = AttnGatingBlock(conv5, g2, 256, '_2')
        # attn2 = Dropout(0.2, name='drop_attn2')(attn2)
        convt2 = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', activation='relu', kernel_initializer=kinit, name='convt2')(up1)
        up2 = concatenate([convt2, attn2], name='up2')

        g3 = UnetGatingSignal(up2, is_batchnorm=True, name='g3')
        attn3 = AttnGatingBlock(conv4, g3, 128, '_3')
        # attn3 = Dropout(0.2, name='drop_attn3')(attn3)
        convt3 = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', activation='relu', kernel_initializer=kinit, name='convt3')(up2)
        up3 = concatenate([convt3, attn3], name='up3')

        g4 = UnetGatingSignal(up3, is_batchnorm=True, name='g4')
        attn4 = AttnGatingBlock(conv3, g4, 128, '_4')
        # attn4 = Dropout(0.2, name='drop_attn4')(attn4)
        convt4 = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', activation='relu', kernel_initializer=kinit, name='convt4')(up3)
        up4 = concatenate([convt4, attn4], name='up4')

        g5 = UnetGatingSignal(up4, is_batchnorm=True, name='g5')
        attn5 = AttnGatingBlock(conv2, g5, 64, '_5')
        attn5 = Dropout(0.2, name='drop_attn5')(attn5)
        convt5 = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', activation='relu', kernel_initializer=kinit, name='convt5')(up4)
        up5 = concatenate([convt5, attn5], name='up5')

        convt6 = Conv2DTranspose(64, (3,3), strides=(2,2), padding='same', activation='relu', kernel_initializer=kinit, name='convt6')(up5)
        up6 = concatenate([convt6, conv1], name='up6')
        conv10 = Conv2D(1, (1, 1), activation='sigmoid',  kernel_initializer=kinit, name='final')(up6)

        model = Model(inputs, conv10)

        # compile model
        model.compile(optimizer = Adam(learning_rate=lr), loss = loss_func, metrics = ['accuracy', f1_m, precision_m, recall_m, dsc])

        if(pretrained_weights):
            model.load_weights(pretrained_weights)

        return model

model = attn_unet(0.001, 'binary_crossentropy',input_size=(512,512,3))
model.summary()

In [ ]:
# callback functions
checkpointer = tf.keras.callbacks.ModelCheckpoint(f"/content/flood_best.keras", # Changed the extension to .keras
                                                 monitor="val_f1_m",
                                                 verbose=1,
                                                 save_best_only=True,
                                                 mode="max")
earlyStopping = tf.keras.callbacks.EarlyStopping(monitor='val_f1_m',
                                                patience=5,
                                                verbose=1,
                                                mode='max')

callbacks = [
    earlyStopping,
    checkpointer
    ]

# model training
history = model.fit(train_dataset,epochs=50,
                    verbose = 1,
                    # validation_split=0.15,
                    validation_data=test_dataset,
                    callbacks=callbacks)

# save the model weights at the end of the training process
model.save(f"/content/flood_save.keras") # Changed the extension to .keras

In [ ]:
fig,((ax11, ax12),(ax13,ax14)) = plt.subplots(2,2,figsize=(10,10))
ax11.plot(history.history['loss'])
ax11.plot(history.history['val_loss'])
ax11.title.set_text('Unet model loss')
ax11.set_ylabel('loss')
ax11.set_xlabel('epoch')
ax11.legend(['train', 'validation'], loc='upper left')

ax12.plot(history.history['precision_m'])
ax12.plot(history.history['val_precision_m'])
ax12.set_title('Unet model precision')
ax12.set_ylabel('precision')
ax12.set_xlabel('epoch')
ax12.legend(['train', 'validation'], loc='upper left')

ax13.plot(history.history['recall_m'])
ax13.plot(history.history['val_recall_m'])
ax13.set_title('Unet model recall')
ax13.set_ylabel('recall')
ax13.set_xlabel('epoch')
ax13.legend(['train', 'validation'], loc='upper left')

ax14.plot(history.history['f1_m'])
ax14.plot(history.history['val_f1_m'])
ax14.set_title('Unet model f1')
ax14.set_ylabel('f1')
ax14.set_xlabel('epoch')
ax14.legend(['train', 'validation'], loc='upper left')

In [ ]:
from keras.models import load_model

# Define the custom loss function before loading the model
custom_objects = {"f1_m": f1_m, 'precision_m': precision_m, 'recall_m': recall_m, 'dsc': dsc}

model = load_model('/content/drive/MyDrive/dl/flood/flood_best.h5', custom_objects=custom_objects)

# pred

In [ ]:
# accuracy, f1_score, precision, recall, dsc = model.evaluate(test_dataset, verbose=0)
loss, accuracy, f1_score, precision, recall, dsc = model.evaluate(test_dataset, verbose=0)
print(loss, accuracy, f1_score, precision, recall, dsc)


In [ ]:
import glob
imgs = glob.glob('/content/*.jpg')

img_test = np.zeros((3, 512, 512, 3))
for index, i in enumerate(imgs):
  img = cv2.imread(i)
  img = cv2.resize(img, (512, 512))
  img = img / 255.0
  # print(img.shape)
  img_test[index] = img


In [ ]:
pred = model.predict(img_test)
pred = np.where(pred > 0.5, 1, 0)
pred.shape

In [ ]:
fig,(axes)= plt.subplots(3, 2, figsize=(12,12))

for i in range(3):
    # Load and display the original image
    axes[i, 0].imshow(img_test[i])
    axes[i, 0].set_title(f'Image {i+1}')
    axes[i, 0].axis('off')

    # Load and display the corresponding prediction
    axes[i, 1].imshow(pred[i])
    axes[i, 1].set_title(f'Prediction {i+1}')
    axes[i, 1].axis('off')

plt.tight_layout()

4. Flooded Region Detection Using SNUNet-CD Model

In [7]:
import os
import torch
import torch.nn as nn
from PIL import Image
import torchvision.transforms as transforms
import numpy as np

# Define NestedUNet and SNUNetCD
class NestedUNet(nn.Module):
    def _init_(self, in_channels=3, out_channels=2):
        super(NestedUNet, self)._init_()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.filters = [64, 128, 256, 512, 1024]

        # Encoding layers
        self.enc1 = self.double_conv(in_channels, self.filters[0])
        self.enc2 = self.double_conv(self.filters[0], self.filters[1])
        self.enc3 = self.double_conv(self.filters[1], self.filters[2])
        self.enc4 = self.double_conv(self.filters[2], self.filters[3])
        self.enc5 = self.double_conv(self.filters[3], self.filters[4])

        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Decoding layers
        self.up4 = nn.ConvTranspose2d(self.filters[4], self.filters[3], kernel_size=2, stride=2)
        self.dec4 = self.double_conv(self.filters[4], self.filters[3])

        self.up3 = nn.ConvTranspose2d(self.filters[3], self.filters[2], kernel_size=2, stride=2)
        self.dec3 = self.double_conv(self.filters[3], self.filters[2])

        self.up2 = nn.ConvTranspose2d(self.filters[2], self.filters[1], kernel_size=2, stride=2)
        self.dec2 = self.double_conv(self.filters[2], self.filters[1])

        self.up1 = nn.ConvTranspose2d(self.filters[1], self.filters[0], kernel_size=2, stride=2)
        self.dec1 = self.double_conv(self.filters[1], self.filters[0])

        # Final output layer
        self.final = nn.Conv2d(self.filters[0], out_channels, kernel_size=1)

    def double_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x1 = self.enc1(x)
        p1 = self.pool(x1)

        x2 = self.enc2(p1)
        p2 = self.pool(x2)

        x3 = self.enc3(p2)
        p3 = self.pool(x3)

        x4 = self.enc4(p3)
        p4 = self.pool(x4)

        x5 = self.enc5(p4)

        up4 = self.up4(x5)
        x4 = torch.cat([up4, x4], dim=1)
        x4 = self.dec4(x4)

        up3 = self.up3(x4)
        x3 = torch.cat([up3, x3], dim=1)
        x3 = self.dec3(x3)

        up2 = self.up2(x3)
        x2 = torch.cat([up2, x2], dim=1)
        x2 = self.dec2(x2)

        up1 = self.up1(x2)
        x1 = torch.cat([up1, x1], dim=1)
        x1 = self.dec1(x1)

        output = self.final(x1)
        return output

class SNUNetCD(nn.Module):
    def _init_(self, in_channels=3, out_channels=1):
        super(SNUNetCD, self)._init_()
        self.unet = NestedUNet(in_channels=in_channels, out_channels=out_channels)

    def forward(self, img1, img2):
        feat1 = self.unet(img1)
        feat2 = self.unet(img2)
        diff = torch.abs(feat1 - feat2)
        return diff

# Function to load and preprocess images
def preprocess_image(image_path, image_size=512):
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor()
    ])
    image = Image.open(image_path).convert('L')  # Ensure grayscale for masks
    return transform(image).unsqueeze(0)  # Add batch dimension

# Function to create the output image
def create_output_image(pre_mask, post_mask, image_size):
    output_image = np.zeros((image_size, image_size, 3), dtype=np.uint8)

    # Existing water bodies (white)
    output_image[(pre_mask == 1) & (post_mask == 1)] = [255, 255, 255]

    # Flooded regions (red)
    output_image[(pre_mask == 0) & (post_mask == 1)] = [255, 0, 0]

    # Non-water bodies (black)
    output_image[(pre_mask == 0) & (post_mask == 0)] = [0, 0, 0]

    return Image.fromarray(output_image)

# Main processing function
def process_folder(pre_flood_dir, post_flood_dir, output_dir, image_size=512):
    os.makedirs(output_dir, exist_ok=True)

    for pre_flood_file in os.listdir(pre_flood_dir):
        pre_flood_path = os.path.join(pre_flood_dir, pre_flood_file)
        post_flood_path = os.path.join(post_flood_dir, pre_flood_file)  # Matching file in post-flood folder
        
        if not os.path.exists(post_flood_path):
            print(f"Skipping {pre_flood_file}: no matching post-flood file found.")
            continue
        
        pre_mask = preprocess_image(pre_flood_path, image_size=image_size)
        post_mask = preprocess_image(post_flood_path, image_size=image_size)

        # Convert masks to binary numpy arrays
        pre_mask = (pre_mask.squeeze().numpy() > 0.5).astype(np.uint8)
        post_mask = (post_mask.squeeze().numpy() > 0.5).astype(np.uint8)

        output_image = create_output_image(pre_mask, post_mask, image_size)

        output_image_path = os.path.join(output_dir, f"{pre_flood_file}")
        output_image.save(output_image_path)
        print(f"Saved output image: {output_image_path}")

# Example usage with folders
if __name__ == "__main__":
    pre_flood_dir = "E:/Sem 5/mini project/water_body/Resized Dataset/Masks"
    post_flood_dir = "E:/Sem 5/mini project/water_body/Resized Dataset/post images masks"
    output_dir = "E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays"

    process_folder(pre_flood_dir, post_flood_dir, output_dir)

Saved output image: E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays\water_body_1.jpg
Saved output image: E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays\water_body_10.jpg
Saved output image: E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays\water_body_100.jpg
Saved output image: E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays\water_body_1000.jpg
Saved output image: E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays\water_body_1002.jpg
Saved output image: E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays\water_body_1003.jpg
Saved output image: E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays\water_body_1004.jpg
Saved output image: E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays\water_body_1006.jpg
Saved output image: E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays\water_body_1008.jpg
Saved output image: E:/Sem 5/mini project/water_body/Resized 

In [ ]:
# Assuming your model is instantiated as 'model'
def save_model(model, path):
    torch.save(model.state_dict(), path)
    print(f"Model saved to {path}")

# Example usage (after training or wherever you want to save the model)
model = SNUNetCD()  # or use NestedUNet() if you're saving that model
save_path = "E:/Sem 5/mini project/snunet_model.pth"  # Specify the path to save the model
save_model(model, save_path)


5. Flooded Region Mapping on Pre-Flood Image

In [9]:
import os
from PIL import Image
import torchvision.transforms as transforms
import numpy as np

# Function to load and preprocess masks to match the target size
def preprocess_mask(mask_path, target_size):
    """
    Preprocess the mask to match the target size.
    """
    # Load the mask
    mask = Image.open(mask_path).convert('RGB')  # Load as RGB to detect red regions
    
    # Resize the mask to the target size (exact dimensions of the pre-flood image)
    mask = mask.resize(target_size, Image.NEAREST)
    
    # Convert to numpy array in HWC format
    mask = np.array(mask)
    return mask

# Function to mark only flooded regions in the original image
def mark_flooded_regions(pre_flood_image, flooded_mask, output_path):
    """
    Mark only the red regions (flooded areas) from the flooded mask onto the original image.
    """
    pre_flood_array = np.array(pre_flood_image)
    
    # Ensure the dimensions of the mask match the dimensions of the pre-flood image
    if flooded_mask.shape[:2] != pre_flood_array.shape[:2]:
        raise ValueError(f"Mask dimensions {flooded_mask.shape[:2]} do not match image dimensions {pre_flood_array.shape[:2]}")
    
    # Identify red regions in the mask (R > G and R > B)
    red_mask = (flooded_mask[:, :, 0] > flooded_mask[:, :, 1]) & (flooded_mask[:, :, 0] > flooded_mask[:, :, 2])
    
    # Mark the corresponding regions in the pre-flood image with red
    red_overlay = [255, 0, 0]  # Red color
    pre_flood_array[red_mask] = red_overlay

    # Save the modified image
    marked_image = Image.fromarray(pre_flood_array)
    marked_image.save(output_path)
    print(f"Saved marked image: {output_path}")

# Main processing function to handle directories
def process_folder(pre_flood_dir, flooded_mask_dir, output_dir):
    """
    Process a folder of images and masks to create overlay images.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    for pre_flood_file in os.listdir(pre_flood_dir):
        pre_flood_path = os.path.join(pre_flood_dir, pre_flood_file)
        flooded_mask_path = os.path.join(flooded_mask_dir, pre_flood_file)  # Matching file in mask folder
        
        # Check for valid file types
        if not pre_flood_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            print(f"Skipping {pre_flood_file}: unsupported file type.")
            continue
        
        if not os.path.exists(flooded_mask_path):
            print(f"Skipping {pre_flood_file}: no matching flooded mask file found.")
            continue
        
        try:
            # Load the pre-flood image
            pre_flood_image = Image.open(pre_flood_path).convert('RGB')
            target_size = pre_flood_image.size  # Get the size of the pre-flood image
            
            # Preprocess the flooded mask to match the size of the pre-flood image
            flooded_mask = preprocess_mask(flooded_mask_path, target_size)
            
            # Save the marked image
            output_path = os.path.join(output_dir, f"{pre_flood_file}")
            mark_flooded_regions(pre_flood_image, flooded_mask, output_path)
        
        except Exception as e:
            print(f"Error processing {pre_flood_file}: {e}")

# Example usage with folders
if __name__ == "__main__":
    pre_flood_dir = "E:/Sem 5/mini project/water_body/Resized Dataset/Images"
    flooded_mask_dir = "E:/Sem 5/mini project/water_body/Resized Dataset/output_overlays"
    output_dir = "E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays"
    
    process_folder(pre_flood_dir, flooded_mask_dir, output_dir)

Saved marked image: E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays\water_body_1.jpg
Saved marked image: E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays\water_body_10.jpg
Saved marked image: E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays\water_body_100.jpg
Saved marked image: E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays\water_body_1000.jpg
Saved marked image: E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays\water_body_1002.jpg
Saved marked image: E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays\water_body_1003.jpg
Saved marked image: E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays\water_body_1004.jpg
Saved marked image: E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays\water_body_1006.jpg
Saved marked image: E:/Sem 5/mini project/water_body/Resized Dataset/flooded_output_overlays\water_bod